In [1]:
from HINGE.evaluation_synthetic import run_synthetic_evaluation_with_dropout

from HINGE.feature_mapping import prepare_expression_features

from sklearn.preprocessing import normalize
import pickle
from pathlib import Path

import matplotlib.pyplot as plt

In [2]:
SAVE_DIR = Path('./hinge_checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
n_gene_comps = 128

In [5]:
with open(SAVE_DIR / 'checkpoint_step1.pkl', "rb") as f:
    data_dict=pickle.load(f)

data_dict

{'RNA_panel1': {'s1': AnnData object with n_obs × n_vars = 6989 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's2': AnnData object with n_obs × n_vars = 8978 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's3': AnnData object with n_obs × n_vars = 449221 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's4': AnnData object with n_obs × n_vars = 7442 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's5': AnnData object with n_obs × n_vars = 16540 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'f

In [6]:
#select which sample to use for the mapping

uni_480_list = [data_dict['HE_panel1'][1]]  # only the first sample for demonstration
uni_5k_list = [data_dict['HE_panel2'][1]]  # only the first sample for demonstration
rna_480_list = [data_dict['RNA_panel1']['s2']]  # only the first sample for demonstration
rna_5k_list = [data_dict['RNA_panel2']['s8']]  # only the first sample for demonstration

In [7]:
uni_480_list

[AnnData object with n_obs × n_vars = 8978 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [8]:

uni_480_normalized_list = []

for adata in uni_480_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_480_normalized_list.append(adata_norm)

In [9]:
uni_5k_normalized_list = []

for adata in uni_5k_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_5k_normalized_list.append(adata_norm)

In [10]:
uni_480_normalized_list

[AnnData object with n_obs × n_vars = 8978 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [11]:
adata_panel1          = rna_480_list[0]
uni_panel1_normed     = uni_480_normalized_list[0]
adata_panel2          = rna_5k_list[0]
uni_panel2_normed     = uni_5k_normalized_list[0]

# run test

In [12]:
# Test on real pair directly:
from HINGE.gene_agreement import compute_gene_agreement_population, compute_gene_agreement_population_2

overlap_genes = list(set(adata_panel1.var_names) & set(adata_panel2.var_names))

gene_stats_v1 = compute_gene_agreement_population(
    adata_panel1, adata_panel2, overlap_genes, verbose=True)

gene_stats_v2 = compute_gene_agreement_population_2(
    adata_panel1, adata_panel2, overlap_genes,
    use_cluster_structure=True, verbose=True)

# Compare: do they select different genes?
# Do the structure scores meaningfully separate genes?
selected_v1 = set(gene_stats_v1[gene_stats_v1.use_for_matching].index)
selected_v2 = set(gene_stats_v2[gene_stats_v2.use_for_matching].index)
print("Only in v1:", selected_v1 - selected_v2)
print("Only in v2:", selected_v2 - selected_v1)

Running compute_gene_agreement_population()   ==================================================

  Population-level gene agreement:
    Total overlapping genes      : 444
    Pass detection (≥5%): 444  (100.0%)
    Pass fold change (≤1.0 log2): 364   (82.0%)
    Pass all filters             : 364  (82.0%)

    Agreement score (passing genes):
      median : 0.743
      mean   : 0.728
      top 10 genes:
           mean_p1   mean_p2    log2fc  detection_p1  detection_p2  agreement_score
FPR1      0.252428  0.253085  0.003749      0.999777      0.999888         0.997958
CDKN1A    0.180180  0.182001  0.014506      0.999889      0.999888         0.992636
MX1       0.156683  0.154755 -0.017867      0.999889      0.999888         0.990956
IFIT1     0.110353  0.112007  0.021472      0.999889      0.999551         0.988986
BLNK      0.249247  0.251985  0.015756      0.984518      0.999888         0.984356
APOBEC3A  0.075172  0.073555 -0.031362      0.999666      0.999888         0.984099
SRC 

KeyboardInterrupt: 

In [21]:
df_dropout, fakes, stats = run_synthetic_evaluation_with_dropout(
    adata_panel1          = adata_panel1,
    uni_panel1_normed     = uni_panel1_normed,
    adata_panel2          = adata_panel2,
    uni_panel2_normed     = uni_panel2_normed,
    #alpha_list            = [0.5, 0.75, 1.0, 1.25, 1.5],
    alpha_list            = [1.0],
    n_seeds_per_alpha     = 2,
    MAX_DIST              = None,
    dropout_frac          = 0.10,
    dropout_mode          = "independent",   # or "symmetric"
    SAMPLE_NAME           = "s1",

    N_PCA_MORPH       = 512,
    n_gene_comps = 512,


    
    evaluate_hinge = True,
    
    leiden_resolution     = 0.5,         # tune if needed
    min_cells_per_cluster = 20,
)

  Computing population-level difference statistics
  Panel1: 8978 cells  |  Panel2: 8899 cells

  Coordinate displacement field:
    median dx        = -0.68 um
    median dy        = -0.19 um
    median NN dist   = 22.87 um
    kept fraction    = 100.0%

  UNI (normalized space, 2048D):
    |delta_mean| median = 0.00264
    delta_std    median = 0.01713

  Expression (444 genes):
    |delta_mean| median = 0.0622
    delta_std    median = 0.1445
  Created s1_t: alpha=1.00  seed=0  xy_method=displacement_field  mean coord shift=40.30 um  n=8978
  Created s1_t: alpha=1.00  seed=1  xy_method=displacement_field  mean coord shift=40.32 um  n=8978

  Total replicates: 2

  s1 → s1_t  [alpha=1.00  rep=1  dropout=10% (independent)]

  Dropout applied  (mode=independent, frac=10%)
    Original n           : 8978
    s1  kept / dropped   : 8080 / 898
    s1t kept / dropped   : 8080 / 898
    Evaluable cells      : 7274  (81.0% of original)
    Non-evaluable (match missing): 806

  --- s1 α=1.00 